# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is primarily a **classification** problem that feeds a **ranking** output.

At the core, I'm predicting a binary label: is this page a decline-recovery
priority (yes/no)? That's classification. But an editor doesn't consume a flat
yes/no list — they work off a queue ordered by priority. So the model's
classification probabilities get sorted into a ranked review queue: classification
under the hood, ranking in production. Same framing as w01's decline_recovery lane.

It's not clustering (I'm not grouping unlabeled pages into unknown segments), and
not pure regression scoring (the outcome I care about is discrete — worth
prioritizing or not — even though the final product is an ordered list).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label — the target column already shipped in the dataset, not something I compute myself.

This is a proxy, not directly observed ground truth. is_declining_label is itself derived from trend_pct (the underlying percentage traffic change), which means it summarizes FlyRank's own definition of "declining" rather than something I independently verified. My model predicts FlyRank's labeling of decline, not "true" Google ranking loss.

One important constraint this creates: trend_pct itself must NEVER be used as a feature — it's the exact number the label is computed from, so including it would be leakage (the answer in disguise, not a real predictive signal). This matters for feature selection starting next week, but it already limits what counts as a legitimate proxy vs. a shortcut.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

if not os.path.exists("flyrank-ml-internship"):
    subprocess.run(["git", "clone", "https://github.com/NAJAM2005/flyrank-ml-internship.git"])
os.chdir("flyrank-ml-internship")
print("Working dir:", os.getcwd())


Working dir: /content/flyrank-ml-internship/flyrank-ml-internship


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@K (specifically Precision@50).

An editor works through a fixed-size queue each sprint, not the full set of
declining pages — so overall accuracy is the wrong metric here (it's inflated by
the base rate, and doesn't reflect how the output is actually used). Precision@K
asks: of the top K pages the model surfaces, what fraction are real priorities?
That's exactly what an editor experiences opening their queue.

This also matches the repo's own reference pipeline, which reports rule-baseline
vs. model performance at P@20 and P@50 — so my results will be directly
comparable to that benchmark once I build a model.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
dummy_scores = df["impressions_90d"].values  # placeholder scoring signal
for k in (20, 50):
    print(f"Precision@{k} using impressions_90d as a naive score: {precision_at_k(dummy_scores, y, k):.3f}")


Precision@20 using impressions_90d as a naive score: 0.450
Precision@50 using impressions_90d as a naive score: 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page.

In [10]:
cols = ["is_declining_label", "trend_direction", "position_tier",
        "impressions_90d", "ctr", "word_count", "days_since_last_update"]
df[cols].head(10)

,is_declining_label,trend_direction,position_tier,impressions_90d,ctr,word_count,days_since_last_update
0,1,down,striking,3803,0.76,3221.0,20
1,1,down,page_3_5,15320,0.05,2481.0,25
2,1,down,page_3_5,12581,0.09,3515.0,20
3,0,stable,page_1,11751,0.49,NaN,22
4,1,down,page_3_5,19140,0.13,2803.0,14
5,1,down,page_1,3970,0.03,3080.0,20
6,1,down,page_1,20,0.00,3059.0,20
7,0,stable,page_3_5,1724,0.06,NaN,22
8,1,down,page_3_5,32574,0.09,3807.0,20
9,1,down,page_1,1240,0.16,NaN,104


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two pieces of evidence, both from Week 1 / the shared walkthrough:

1. No single strong signal separates decliners. Median word count is nearly
   identical for declining vs. growing pages — a rule like "flag if word_count
   < X" would misfire constantly.
2. Signals interact rather than acting independently. CTR varies sharply by
   position_tier (roughly 0.35 near page_1 down to 0.06 "deep") — so a flat CTR
   threshold means something different depending on where a page already ranks.

A hand rule trying to capture "low CTR relative to position, combined with
staleness, combined with enough impression volume to matter" needs a dozen
nested conditions before it's useful — each added condition is a guess about
an interaction a model can instead learn directly from data.

In [11]:
interaction = df.groupby(["position_tier"])["ctr"].mean().sort_values(ascending=False)
print(interaction.round(4))

wc = df.groupby("is_declining_label")["word_count"].median()
print("\nMedian word count by is_declining_label:")
print(wc.round(0))

position_tier
top_3       1.4836
page_1      0.6525
striking    0.3232
page_3_5    0.2225
deep        0.1502
Name: ctr, dtype: float64

Median word count by is_declining_label:
is_declining_label
0    2839.0
1    2909.0
Name: word_count, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.